# FIRE13 EVALUATION RECOVERY ONLY

Internet OFF. GPU T4, une seule carte visible. Attacher les onze partitions existantes p00-p10 et un dataset prive contenant TRAINING-SAFE + son .sha256. Aucun entrainement. Aucun pixel 2021. Seuil principal 0,78 et criteres historiques inchanges. Les deux autorisations ci-dessous sont nouvelles. Ne jamais supprimer un verrou pour recommencer.


In [ ]:
RUN_GPU_PREFLIGHT = False
RUN_EVALUATION = False
import os,sys
if 'torch' in sys.modules: raise RuntimeError('Restart kernel before CUDA configuration')
os.environ['CUDA_VISIBLE_DEVICES']='0'


In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,json
DATA=base64.b64decode('...')
assert hashlib.sha256(DATA).hexdigest()=='3a13ab0552851abe6a708e5c6e0180a9e22bb68d4cfb22c5b307f073bc73211c'
ROOT=Path('/kaggle/working/geofirex-fire13-recovery-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(DATA)) as z:
    for n in z.namelist():
        assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
from geofirex import fire13_recovery as r
c=r.c
import importlib
for module in c.read(ROOT/r.REPORT/'dependency_audit.json')['modules']:
    importlib.import_module(module)
print(r.identity(ROOT)['checkpoint_sha256'])


In [ ]:
EXPECTED={v['map_sha256'] for v in c.read(ROOT/c.old.REPORT/'partitions.json').values()}
MAPS=[p for p in Path('/kaggle/input').rglob('mount-map.json') if c.sha256(p) in EXPECTED]
assert len(MAPS)==11
print(c.preflight(ROOT,MAPS))
SOURCES=list(Path('/kaggle/input').rglob('B2_C13_FIRE_V2_seed42-TRAINING-SAFE-c1145600.zip'))+list(Path('/kaggle/input').rglob('B2_C13_FIRE_V2_seed42-TRAINING-SAFE-c1145600.bin'))
assert len(SOURCES)==1, 'Attach exactly one original TRAINING-SAFE archive (zip or unchanged bin copy)'
ARCHIVE=SOURCES[0]
SIDECAR=ARCHIVE.with_suffix('.sha256')
assert SIDECAR.is_file()
SOURCE=Path('/kaggle/working/fire13-restored-training-safe')
OUT=Path('/kaggle/working')/r.OUT_NAME
print(r.resource_preflight(ROOT,ARCHIVE,OUT.parent))


Renseigner les observations ACTUELLES de quota et session dans la cellule suivante. Reserve conservative : 300 minutes et 10 heures-carte (deux T4 potentiellement facturees), 16 GiB RAM totale / 8 GiB disponibles, au moins 18 GiB disque plus restauration/copies. Delai inference + AP + comparaisons : 8100 secondes. Les reserves ne garantissent pas la duree. Activer explicitement CREATE_NEW_AUTHORIZATIONS une seule fois, puis RUN_GPU_PREFLIGHT et enfin RUN_EVALUATION. Ne pas reutiliser le grant du fit.


In [ ]:
CREATE_NEW_AUTHORIZATIONS = False
OPERATOR = ''
SESSION_REMAINING_MINUTES = None
QUOTA_REMAINING_CARD_HOURS = None
LOCAL_BACKUP = r'C:\Users\nourb\Projects\GeoFire-X-2026\artifacts\fire13'
EXTERNAL_BACKUP = r'D:\GeoFire-X\GeoFire-X-V2'
GPU_AUTH=OUT.parent/'fire13_recovery_gpu_authorization.json'
EVAL_AUTH=OUT.parent/'fire13_recovery_evaluate_authorization.json'
if CREATE_NEW_AUTHORIZATIONS:
    from datetime import datetime,timezone
    import uuid
    assert OPERATOR.strip() and SESSION_REMAINING_MINUTES is not None and QUOTA_REMAINING_CARD_HOURS is not None
    now=datetime.now(timezone.utc).isoformat()
    for action,path in [('gpu-preflight',GPU_AUTH),('fire13-evaluate',EVAL_AUTH)]:
        a=c.read(ROOT/r.REPORT/('authorization_'+action+'_NEW_DISABLED.json'))
        a.update(authorized=True,authorized_by=OPERATOR,authorized_at=now,quota_checked_at=now,
            authorization_id=str(uuid.uuid4()),session_remaining_minutes=SESSION_REMAINING_MINUTES,
            quota_remaining_card_hours=QUOTA_REMAINING_CARD_HOURS,card_hours_reserved=10.,
            local_backup_destination=LOCAL_BACKUP,external_backup_destination=EXTERNAL_BACKUP)
        c.save(path,a)
        r.authorize(ROOT,path,action,OUT.parent)


In [ ]:
if RUN_GPU_PREFLIGHT:
    r.authorize(ROOT,GPU_AUTH,'gpu-preflight',OUT.parent)
    print(c.preflight(ROOT,MAPS,OUT.parent,'cuda',gpu=True))
else: print('GPU preflight disabled')


In [ ]:
if RUN_EVALUATION:
    assert RUN_GPU_PREFLIGHT
    r.authorize(ROOT,EVAL_AUTH,'fire13-evaluate',OUT.parent)
    r.resource_preflight(ROOT,ARCHIVE,OUT.parent)
    r.restore(ROOT,ARCHIVE,SIDECAR,SOURCE)
    print(r.verify_final(ROOT,SOURCE)[1])
    r.recover(ROOT,MAPS,SOURCE,OUT,EVAL_AUTH,GPU_AUTH,ARCHIVE)
else: print('Evaluation disabled')


In [ ]:
from IPython.display import display,FileLink
for p in sorted(OUT.parent.glob(r.OUT_NAME+'-*.zip')):
    print(c.old.verify_zip(p,p.with_suffix('.sha256').read_text().split()[0]))
    display(FileLink(str(p)));display(FileLink(str(p.with_suffix('.sha256'))))
for p in OUT.parent.glob('fire13-evaluate-*.lock.json'): display(FileLink(str(p)))
if (OUT/'evaluation/complete.json').exists():
    print(json.dumps(c.read(OUT/'evaluation/paired_comparison/FIRE13_acceptance.json'),indent=2))


Sauvegarder ZIP final, SHA-256 et verrou de consommation dans les deux destinations, puis verifier les hashes. Les chemins Windows sont des destinations de telechargement, pas des montages accessibles a Kaggle. Conserver les archives originales et le verrou du fit sans modification. Conserver ce notebook execute et le quota reel avant/apres session. Aucun nouvel essai apres consommation sans revue explicite ; le verrou local ne peut pas empecher un nouvel essai dans une autre session Kaggle vierge. Ne pas relancer cette evaluation dans une autre session. Aucune integration Studio automatique.
